# a. Lagrange interpolation

In [1]:
def lagrange_interpolation(x, x_known, f_known):
    """
    Evaluate the Lagrange interpolating polynomial at x.

    x:  point to evaluate
    x_lnown list of known x-coordinates (nodes)
    f_known: list of known function values at x_known
    """
    # polynomial degree = points - 1
    n = len(x_known) - 1

    result = 0.0

    # loop over L_j(x)
    for j in range(n + 1): 
        pj = 1.0

        # loop over all other nodes k!=j
        for k in range(n + 1):
            if k != j:
                pj *= (x - x_known[k]) / (x_known[j] - x_known[k])

        result += f_known[j] * pj

    return result


In [ ]:
xi = [0.0, 0.4, 0.8, 1.2, 1.6]
fi = [0.0, 0.428392, 0.742101, 0.910314, 0.970348]

exact = {0.3: 0.328627, 0.5: 0.520500}

# loop over the 2 test points
# compute interpolant
# compare to the known exact value
for i in (0.3, 0.5):
    interpol = lagrange_interpolation(i, xi, fi)          # interpolated value
    err = interpol - exact[i]                             # signed error vs. exact
    print(f"f({i}) interpolated = {interpol:.6f};   exact = {exact[i]:.6f};   error = {err:.6f}")

f(0.3) interpolated = 0.329345;   exact = 0.328627;   error = 0.000718
f(0.5) interpolated = 0.519939;   exact = 0.520500;   error = -0.000561


# b. Aitken vs Lagrange

In [ ]:
def lagrange_interpolation_counted(x, x_known, f_known):
    """
    Same as lagrange_interpolation, but also counts arithmetic operations.

    x:  point to evaluate
    x_known: list of known x-coordinates (nodes)
    f_known: list of known function values at x_known
    """
    n = len(x_known) - 1
    ops = {'add': 0, 'sub': 0, 'mul': 0, 'div': 0}
    result = 0.0

    # loop over L_j(x)
    for j in range(n + 1):
        pj = 1.0

        # loop over all other nodes k != j
        for k in range(n + 1):
            if k != j:
                num = x - x_known[k];              ops['sub'] += 1
                den = x_known[j] - x_known[k];      ops['sub'] += 1
                pj *= num / den;                    ops['div'] += 1; ops['mul'] += 1

        result += f_known[j] * pj;                  ops['mul'] += 1; ops['add'] += 1

    return result, ops

In [ ]:
def aitken_interpolation_counted(x, x_known, f_known):
    """
    Aitken/Neville recursive interpolation, counting arithmetic operations.

    x:  point to evaluate
    x_known: list of known x-coordinates (nodes)
    f_known: list of known function values at x_known
    """
    n = len(x_known) - 1
    ops = {'add': 0, 'sub': 0, 'mul': 0, 'div': 0}
    ft = list(f_known)   # working copy, updated column by column

    # build the triangular table one level i at a time
    for i in range(n):
        for j in range(n - i):
            a = x - x_known[j]                       ; ops['sub'] += 1
            b = x_known[i+j+1] - x_known[j]          ; ops['sub'] += 1   # reused via sign flip below
            c = x - x_known[i+j+1]                    ; ops['sub'] += 1
            term1 = (a/b) * ft[j+1]                   ; ops['div'] += 1; ops['mul'] += 1
            term2 = (-c/b) * ft[j]                     ; ops['div'] += 1; ops['mul'] += 1
            ft[j] = term1 + term2                      ; ops['add'] += 1

    return ft[0], ops

In [ ]:
print(f"{'n':>4} {'pts':>5} {'Lagrange':>10} {'Aitken':>10} {'Aitken/Lagrange':>16}")
for n in (4, 8, 12, 16, 20, 30, 40, 80):
    x_known = [i/n for i in range(n+1)]
    f_known = [1/(1+xx**2) for xx in x_known]
    x_test = 0.37

    v1, o1 = lagrange_interpolation_counted(x_test, x_known, f_known)
    v2, o2 = aitken_interpolation_counted(x_test, x_known, f_known)
    tot1, tot2 = sum(o1.values()), sum(o2.values())

    print(f"{n:4d} {n+1:5d} {tot1:10d} {tot2:10d} {tot2/tot1:14.2f}   match={abs(v1-v2)<1e-9}")

# c. Rounding error in direct Aitken

In [ ]:
import numpy as np
from fractions import Fraction as F

In [ ]:
def lagrange_interpolation_f32(x, x_known, f_known):
    """
    Direct Lagrange interpolation, forced to single-precision arithmetic
    to make rounding effects visible at moderate n.

    x:  point to evaluate
    x_known: list of known x-coordinates (nodes)
    f_known: list of known function values at x_known
    """
    n = len(x_known) - 1
    x = np.float32(x)
    x_known = [np.float32(v) for v in x_known]
    f_known = [np.float32(v) for v in f_known]

    s = np.float32(0.0)
    for j in range(n + 1):
        p = np.float32(1.0)
        for k in range(n + 1):
            if k != j:
                p = np.float32(p * ((x - x_known[k]) / (x_known[j] - x_known[k])))
        s = np.float32(s + f_known[j] * p)
    return float(s)

In [ ]:
def aitken_interpolation_f32(x, x_known, f_known):
    """
    Direct Aitken/Neville recursion, forced to single-precision arithmetic
    to make rounding effects visible at moderate n.

    x:  point to evaluate
    x_known: list of known x-coordinates (nodes)
    f_known: list of known function values at x_known
    """
    n = len(x_known) - 1
    x = np.float32(x)
    x_known = [np.float32(v) for v in x_known]
    ft = [np.float32(v) for v in f_known]   # working copy

    for i in range(n):
        for j in range(n - i):
            t1 = np.float32((x - x_known[j]) / (x_known[i+j+1] - x_known[j]) * ft[j+1])
            t2 = np.float32((x - x_known[i+j+1]) / (x_known[j] - x_known[i+j+1]) * ft[j])
            ft[j] = np.float32(t1 + t2)
    return float(ft[0])

In [ ]:
def lagrange_interpolation_exact(x, x_known, f_known):
    """
    Same formula as lagrange_interpolation, but using exact rational
    (Fraction) arithmetic - zero rounding error - to serve as ground truth.

    x:  point to evaluate (Fraction)
    x_known: list of known x-coordinates (Fractions)
    f_known: list of known function values at x_known (Fractions)
    """
    n = len(x_known) - 1
    s = F(0)
    for j in range(n + 1):
        p = F(1)
        for k in range(n + 1):
            if k != j:
                p *= (x - x_known[k]) / (x_known[j] - x_known[k])
        s += f_known[j] * p
    return s

In [ ]:
# Test function: f(x) = 1/(1+25x^2) on equally-spaced nodes in [-1, 1],
# evaluated at x = 0.95. Nodes are chosen as rationals so the "exact"
# reference has zero rounding error.
print(f"{'n':>4} {'Lagrange err':>14} {'Aitken err':>12} {'Aitken/Lagrange':>16}")
for n in (15, 18, 20, 24):
    x_known_frac = [F(-1) + F(2*i, n) for i in range(n + 1)]
    f_known_frac = [F(1) / (1 + 25*xx**2) for xx in x_known_frac]
    x_known = [float(v) for v in x_known_frac]
    f_known = [float(v) for v in f_known_frac]

    x_frac = F(19, 20)     # x = 0.95, exactly
    x_test = float(x_frac)

    exact = float(lagrange_interpolation_exact(x_frac, x_known_frac, f_known_frac))
    lag32 = lagrange_interpolation_f32(x_test, x_known, f_known)
    ait32 = aitken_interpolation_f32(x_test, x_known, f_known)

    err_lag, err_ait = abs(lag32 - exact), abs(ait32 - exact)
    print(f"{n:4d} {err_lag:14.3e} {err_ait:12.3e} {err_ait/err_lag:16.2f}")